### Імпорт бібліотек, встановлення налаштувань відображення, завантаження файлу

In [175]:
import pandas as pd
import numpy as np
import re

In [176]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

In [177]:
df = pd.read_csv("D:\\my_projects\\kursah\\data\\raw\\result_test_5000.csv")

In [178]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   price_UAH           5000 non-null   str  
 1   price_USD           5000 non-null   str  
 2   title               5000 non-null   str  
 3   location_list       5000 non-null   str  
 4   details_list        5000 non-null   str  
 5   advantages_list     5000 non-null   str  
 6   description         5000 non-null   str  
 7   created_at          5000 non-null   str  
 8   facilities_list     5000 non-null   str  
 9   technically_tested  5000 non-null   str  
 10  seller_name         5000 non-null   str  
 11  seller_trust        5000 non-null   str  
 12  city                5000 non-null   str  
 13  plus_text           5000 non-null   str  
 14  url                 5000 non-null   str  
 15  page                5000 non-null   int64
dtypes: int64(1), str(15)
memory usage: 625.1 KB


## 1. Очистка і формування стовпчиків для аналізу

### Загальні стовпці з ціною, id, кількістю кімнат

In [179]:
df['id'] = df['title'].str.extract(r'ID (\d+)').astype(int)
df['rooms'] = df['title'].str.extract(r'(\d+)к квартири').astype(int)
df['price_UAH'] = df['price_UAH'].str.replace(' ', '').str.extract(r'(\d+)').astype(int)
df['price_USD'] = df['price_USD'].str.replace(' ', '').str.extract(r'(\d+)').astype(int)

### Стовпець дати

In [180]:
months = {
    'січ': '01', 'лют': '02', 'бер': '03', 'кві': '04',
    'тра': '05', 'чер': '06', 'лип': '07', 'сер': '08',
    'вер': '09', 'жов': '10', 'лис': '11', 'гру': '12'
}
extracted = df['created_at'].str.extract(r'(\d+)\s+([а-яіїєґ]+)\.*\s*(\d{4})?')
extracted.columns = ['day', 'month_abbr', 'year']
current_year = str(pd.Timestamp.now().year)
extracted['year'] = extracted['year'].fillna(current_year)
extracted['month'] = extracted['month_abbr'].map(months)

df['date'] = pd.to_datetime(
    extracted['day'] + '.' + extracted['month'] + '.' + extracted['year'],
    format='%d.%m.%Y'
)

### Створення стовпців з локацією

In [181]:
df['city_name'] = df['city'].str.split(',').str[2].str.strip().astype(str)
df['district'] = df['location_list'].str.extract(r'район\s+([^|]+)')[0].str.strip()
df['metro'] = df['location_list'].str.extract(r'метро\s+([^|]+)')[0].str.strip()
df['address'] = df['title'].str.extract(r'на\s+((?:вул|просп|бул|пров|пл|шосе|наб)\.?\s+.+?)(?=\s*•)')[0].str.strip()

### Створення стовпців з інформацією про продавця

In [182]:
df['seller'] = df['seller_name'].str.extract(r'\n([А-ЯІЇЄҐа-яA-Za-z][^\n\d]+?)(?:\n|$)')[0].str.strip()

def experience_to_months(text):
    if pd.isna(text):
        return 0
    
    years  = re.search(r'(\d+)\+?\s*рок|(\d+)\+?\s*рік', text)
    months = re.search(r'(\d+)\s*місяц', text)
    days   = re.search(r'(\d+)\s*дн', text)
    
    total = 0
    if years:
        total += int(years.group(1) or years.group(2)) * 12
    if months:
        total += int(months.group(1))
    if days:
        total += int(days.group(1)) / 30
    
    return total

def categorize(months):
    if months < 12:
        return 'Новачок'        # < 1 року
    elif months < 36:
        return 'Досвідчений'     # 1–3 роки
    elif months < 60:
        return 'Професіонал'  # 3–5 років
    else:
        return 'Експерт'      # 5+ років
    
df['experience_months'] = df['seller_trust'].apply(experience_to_months).round().astype(int)
df['seller_level'] = df['experience_months'].apply(categorize)

print(df[['experience_months', 'seller_level']])

      experience_months seller_level
0                    60      Експерт
1                    60      Експерт
2                    60      Експерт
3                    60      Експерт
4                    60      Експерт
5                    60      Експерт
6                    60      Експерт
7                    60      Експерт
8                    60      Експерт
9                     5      Новачок
10                   60      Експерт
11                   60      Експерт
12                   60      Експерт
13                   60      Експерт
14                   60      Експерт
15                   60      Експерт
16                   60      Експерт
17                   33  Досвідчений
18                   60      Експерт
19                   34  Досвідчений
20                   34  Досвідчений
21                   60      Експерт
22                   60      Експерт
23                   60      Експерт
24                   60      Експерт
25                   32  Досвідчений
2

### Створення стовпців по площі 

In [183]:
df['area_total']   = df['details_list'].str.extract(r'[Зз]агальна площа\s+([\d.]+)')[0]
df['area_living']  = df['details_list'].str.extract(r'житлова\s+([\d.]+)')[0]
df['area_kitchen'] = df['details_list'].str.extract(r'кухня\s+([\d.]+)')[0]

for col in ['area_total', 'area_living', 'area_kitchen']:
    df[col] = df[col].astype(float)

print(df[['area_total', 'area_living', 'area_kitchen']])

      area_total  area_living  area_kitchen
0          27.00        16.00          6.00
1          87.00        60.00         20.00
2         140.00        55.40         34.20
3          40.00          NaN         18.80
4          51.20        14.70         23.60
5          44.10          NaN           NaN
6          21.00        14.00          4.00
7          26.00        14.00          4.00
8          46.00          NaN           NaN
9          30.00        16.00          6.00
10         56.00          NaN           NaN
11         44.00          NaN           NaN
12         46.00          NaN           NaN
13         90.00        49.00         12.50
14        125.00        80.00         12.00
15        198.00        70.00         22.00
16         74.00          NaN           NaN
17        125.20        66.70         13.60
18         47.00          NaN           NaN
19        150.00       120.00         17.00
20        150.00       120.00         17.00
21         36.00        18.00   

### Створення стовпців поверху квартири та поверховості будинку

In [184]:
df['floor_current'] = df['technically_tested'].str.extract(r'(\d+)\s+поверх з')[0].astype('Int64')
df['floor_total']   = df['technically_tested'].str.extract(r'поверх з\s+(\d+)')[0].astype('Int64')

print(df[['floor_current', 'floor_total']])

      floor_current  floor_total
0                 3            4
1                 5           16
2                 7            8
3                 4            9
4                23           26
5                10           16
6                 2            3
7                 1            2
8                 4           15
9                 2            2
10                4           16
11               11           15
12                9           10
13               10           26
14                7            7
15                3            5
16               11           30
17               11           12
18                7           18
19                2            6
20                2            6
21               10           10
22                6           12
23                3            9
24               11           18
25               14           25
26                1            4
27                2           10
28               12           16
29        

### Створення бінарних стовпців по особливостям планування

In [185]:
df['kitchen_studio'] = df['plus_text'].str.extract(r'(Кухня-студія)', flags=re.IGNORECASE)
df['penthouse']      = df['plus_text'].str.extract(r'(Пентхаус)', flags=re.IGNORECASE)
df['multilevel']     = df['plus_text'].str.extract(r'(Багаторівнева)', flags=re.IGNORECASE)

for col in ['kitchen_studio', 'penthouse', 'multilevel']:
    df[col] = df[col].notna().astype(int)

print(df[['kitchen_studio', 'penthouse', 'multilevel']])

      kitchen_studio  penthouse  multilevel
0                  1          0           0
1                  1          0           0
2                  1          1           1
3                  1          0           0
4                  1          0           0
5                  1          0           0
6                  0          0           0
7                  1          0           0
8                  0          0           0
9                  0          0           0
10                 0          0           0
11                 0          0           0
12                 0          0           0
13                 0          0           0
14                 1          1           1
15                 1          0           0
16                 1          0           0
17                 0          0           0
18                 1          0           0
19                 0          0           0
20                 0          0           0
21                 0          0 

### Створення стовпця про наявність укриття 

In [186]:
df['shelter'] = df['advantages_list'].str.extract(r'((?:У|у)криття[^|]+)')[0].str.strip()

def extract_shelter_context(text):
    if pd.isna(text):
        return None
    match = re.search(r'(\S+\s+){0,2}\S*[Уу]крит\S*(\s+\S+){0,2}', text)
    return match.group().strip() if match else None

df['shelter_context'] = df['description'].apply(extract_shelter_context)
df['shelter_plus'] = df['plus_text'].apply(extract_shelter_context)

print(df['shelter'])

0       Укриття в будинку
1                     NaN
2                     NaN
3                     NaN
4                     NaN
5                     NaN
6                     NaN
7                     NaN
8                     NaN
9                     NaN
10                    NaN
11                    NaN
12                    NaN
13                    NaN
14                    NaN
15                    NaN
16                    NaN
17                    NaN
18                    NaN
19      Укриття в будинку
20      Укриття в будинку
21                    NaN
22                    NaN
23                    NaN
24                    NaN
25                    NaN
26                    NaN
27                    NaN
28                    NaN
29                    NaN
30                    NaN
31                    NaN
32                    NaN
33                    NaN
34                    NaN
35                    NaN
36      Укриття в будинку
37                    NaN
38      Укри

Зведення до єдиної бінарної колонки по наявності укриття:

In [187]:
df['shelter_plus'] = df['shelter_plus'].str.contains('укриття', case=False, na=False).astype(int)
df['shelter_context'] = df['shelter_context'].str.contains('укриття', case=False, na=False).astype(int)
df['shelter'] = df['shelter'].str.contains('укриття', case=False, na=False).astype(int)

shelter_columns = ['shelter_plus', 'shelter_context', 'shelter']
df['has_shelter'] = df[shelter_columns].any(axis=1).astype(int)


# print(df[['shelter_plus', 'shelter_context', 'shelter']])
print(df['has_shelter'].value_counts())

has_shelter
0    3773
1    1227
Name: count, dtype: int64


### Створення стовпця дозволу проживання з тваринами

In [188]:
df['pets'] = df['facilities_list'].str.extract(r'((?:[Мм]ожна|[Нн]е можна|[Бб]ез)\s+з?\s*тварин[^|]*)')[0].str.strip()

def extract_bober_context(text):
    if pd.isna(text):
        return None
    match = re.search(r'(\S+\s+){0,2}\S*[Тт]варин\S*(\s+\S+){0,2}', text)
    return match.group().strip() if match else None

df['pets_desc'] = df['description'].apply(extract_bober_context)

Зведення до єдиної бінарної колонки по дозволу проживання з тваринами:

In [189]:
df['pets'] = df['pets'].fillna('Не вказано')

def classify_pets_desc(text):
    if pd.isna(text):
        return None
    text_lower = text.lower()
    
    if re.search(r'без тварин|не можна|заборон|неможлив|не приймаємо тварин|не розглядаєть|не розглядаємо', text_lower):
        return 'не можна'
    
    if re.search(r'тварин|тваринк', text_lower):
        return 'можна'
    
    return None

df['pets_classified'] = df['pets_desc'].apply(classify_pets_desc)

df['pets'] = df['pets'].fillna(df['pets_classified']).fillna('не вказано')


print(df['pets'].value_counts())

pets
Не вказано           2573
Без тварин           1519
Можна з тваринами     908
Name: count, dtype: int64


### Створення стовпців забезпеченості при відсутності електроенергії.   

In [190]:
features = {
    'no_light_internet':    'є інтернет',
    'no_light_mobile_connection':      'є мобільний зв\'язок',
    'no_light_water':       'є водопостачання',
    'no_light_heating':     'працює опалення',
    'no_light_gas':         'є газ',
    'no_light_elevator':    'працює ліфт'
}

for col, keyword in features.items():
    df[col] = df['plus_text'].str.contains(keyword, case=False, na=False).astype(int)

print(df[['no_light_internet', 'no_light_mobile_connection', 'no_light_water', 'no_light_heating', 'no_light_gas',
           'no_light_elevator']])

      no_light_internet  no_light_mobile_connection  no_light_water  \
0                     1                           0               0   
1                     1                           0               1   
2                     1                           0               1   
3                     0                           0               0   
4                     1                           1               1   
5                     0                           1               1   
6                     1                           1               1   
7                     1                           1               1   
8                     0                           0               0   
9                     1                           0               0   
10                    0                           0               0   
11                    0                           0               0   
12                    0                           0               0   
13    

### **Створення кінцевого датасету**

In [191]:
cl_df = df[['id', 'rooms', 'price_UAH', 'price_USD', 'date', 'city_name', 'district', 'metro', 'address', 'seller',
            'experience_months', 'seller_level', 'area_total', 'area_living', 'area_kitchen', 'floor_current', 
            'floor_total', 'kitchen_studio', 'penthouse', 'multilevel', 'has_shelter', 'pets', 'no_light_internet',
            'no_light_mobile_connection', 'no_light_water', 'no_light_heating', 'no_light_gas', 'no_light_elevator',
            'url']].copy()

print(cl_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 29 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   id                          5000 non-null   int64         
 1   rooms                       5000 non-null   int64         
 2   price_UAH                   5000 non-null   int64         
 3   price_USD                   5000 non-null   int64         
 4   date                        5000 non-null   datetime64[us]
 5   city_name                   4963 non-null   str           
 6   district                    4383 non-null   str           
 7   metro                       1310 non-null   str           
 8   address                     4897 non-null   str           
 9   seller                      5000 non-null   str           
 10  experience_months           5000 non-null   int64         
 11  seller_level                5000 non-null   str           
 12  are

In [192]:
print(cl_df.head(10))

         id  rooms  price_UAH  price_USD       date city_name       district  \
0  33683279      1       8000        181 2025-11-11      Київ     Дарницький   
1  31091475      3      48510       1100 2026-02-08      Київ  Святошинський   
2  33543548      3      85000       1928 2025-09-29      Київ          Поділ   
3  34035058      1      37485        850 2026-03-06      Київ          Нивки   
4  33961934      1      34178        775 2026-02-15      Київ     Солом'янка   
5  34014872      1      35280        800 2026-03-01   Вінниця        Кумбари   
6  34064125      1       7700        175 2026-03-16      Київ   Дніпровський   
7  34050354      1       7500        170 2026-03-13      Київ   Дніпровський   
8  33916314      1      37485        850 2026-02-19   Вінниця          Центр   
9  33541826      1       7000        159 2025-12-13     Одеса      Київський   

               metro                                   address  \
0     Червоний хутір              вул. Бориспільська 

## 2. EDA (Exploratory Data Analysis)

In [193]:
cl_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 29 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   id                          5000 non-null   int64         
 1   rooms                       5000 non-null   int64         
 2   price_UAH                   5000 non-null   int64         
 3   price_USD                   5000 non-null   int64         
 4   date                        5000 non-null   datetime64[us]
 5   city_name                   4963 non-null   str           
 6   district                    4383 non-null   str           
 7   metro                       1310 non-null   str           
 8   address                     4897 non-null   str           
 9   seller                      5000 non-null   str           
 10  experience_months           5000 non-null   int64         
 11  seller_level                5000 non-null   str           
 12  are

Спочатку перевіряємо повністю дублюючі рядки і видаляємо дублікати. 

In [194]:
n_duplicates = cl_df.duplicated().sum()
print(f"Кількість повних дублікатів: {n_duplicates}")
print(f"Це {n_duplicates / len(cl_df) * 100:.2f}% від усіх рядків ({len(cl_df)} загалом)")

cl_df = cl_df.drop_duplicates()
print(f"\nПісля видалення: {len(cl_df)} рядків залишилось")

Кількість повних дублікатів: 68
Це 1.36% від усіх рядків (5000 загалом)



Після видалення: 4932 рядків залишилось


Далі перевіряємо і видаляємо дублікати по колонці `id`

In [195]:
n_duplicates_id = cl_df.duplicated(subset=['id']).sum()
print(f"Кількість дублікатів по колонці id: {n_duplicates_id}")
print(f"Це {n_duplicates_id / len(cl_df) * 100:.2f}% від усіх рядків ({len(cl_df)} загалом)")

cl_df = cl_df.drop_duplicates(subset=['id'])
print(f"\nПісля видалення: {len(cl_df)} рядків залишилось")

Кількість дублікатів по колонці id: 4
Це 0.08% від усіх рядків (4932 загалом)

Після видалення: 4928 рядків залишилось


Просто видаляємо колонки з відсутнім поверхом, їх небагато. Це може бути через неперевірену квартиру або збій - обидва варіанти нам не підходять.

In [196]:
print(f"До: {len(cl_df)}")
cl_df = cl_df.dropna(subset=['floor_current'])
print(f"Після: {len(cl_df)}")

До: 4928
Після: 4918


В наступних комірках заповнюємо пропущені значення медіаною групи і загальною. Це так звана Імпутація. Оскільки це площа кухні і житлова за орієнтир беремо кількість кімнат. Також порівнюємо медіану і середнє до та після заповнення.

In [197]:
print(cl_df['area_kitchen'].median())
print(cl_df['area_kitchen'].mean())
print(cl_df['area_living'].median())
print(cl_df['area_living'].mean())

11.8
12.722199119373776
28.0
32.27910414949162


In [198]:
# Заповнюємо площу медіаною на основі кількості кімнат
cl_df['area_living'] = cl_df['area_living'].fillna(
    cl_df.groupby('rooms')['area_living'].transform('median')
)


cl_df['area_kitchen'] = cl_df['area_kitchen'].fillna(
    cl_df.groupby('rooms')['area_kitchen'].transform('median')
)

# Залишок заповнюємо просто загальною медіаною
cl_df['area_kitchen'] = cl_df['area_kitchen'].fillna(cl_df['area_kitchen'].median())
cl_df['area_living'] = cl_df['area_living'].fillna(cl_df['area_living'].median())

In [199]:
print(cl_df['area_kitchen'].median())
print(cl_df['area_kitchen'].mean())
print(cl_df['area_living'].median())
print(cl_df['area_living'].mean())

11.5
12.4897417649451
28.0
31.16693981293209


Після заповнення пропусків спостерігаються незначні зміни у середньому значення та медіані, проте вони не спотворюють загальні дані і знаходяться в межах дозволеного. 

Також далі видаляємо рядки без адреси і без міста, їх теж небагато.

In [200]:
cl_df = cl_df.dropna(subset=['address'])
cl_df = cl_df.dropna(subset=['city_name'])

Ще заповнюємо відсутні значення у стовпці метро 'Немає метро' і створюємо для майбутніх дій бінарний стовпець 'has_metro'.
Відсутні значення у стовпці район також заповнюємо 'Невідомий район'.

In [201]:
cl_df['metro'] = cl_df['metro'].fillna('Немає метро')
cl_df['has_metro'] = (cl_df['metro'] != 'Немає метро').astype(int)


cl_df['district'] = cl_df['district'].fillna('Невідомий район')

Дивимося на результат заповнення порожніх комірок: маємо 4781 з 5000, тобто це 95,62% - можна сказати чудовий результат після парсера сайта і очищення даних за допомогою регулярних виразів.

In [202]:
cl_df.info()

<class 'pandas.DataFrame'>
Index: 4781 entries, 0 to 4999
Data columns (total 30 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   id                          4781 non-null   int64         
 1   rooms                       4781 non-null   int64         
 2   price_UAH                   4781 non-null   int64         
 3   price_USD                   4781 non-null   int64         
 4   date                        4781 non-null   datetime64[us]
 5   city_name                   4781 non-null   str           
 6   district                    4781 non-null   str           
 7   metro                       4781 non-null   str           
 8   address                     4781 non-null   str           
 9   seller                      4781 non-null   str           
 10  experience_months           4781 non-null   int64         
 11  seller_level                4781 non-null   str           
 12  area_tot